In [ ]:
"""Script to download monthly stock price data from Yahoo Finance for a list of tickers. Saves the data to a CSV file."""

import pandas as pd
import numpy as np
import requests
from datetime import datetime
import time
import json

def get_stock_universe(file_path: str, n: int = 645):
    """Load the first n tickers from the 'ticker' column in a CSV file and return a list containing n tickers."""
    try:
        df = pd.read_csv(file_path)
        if 'ticker' not in df.columns:
            raise KeyError("CSV file must contain a column named 'ticker'")
        
        tickers = df['ticker'].dropna().astype(str).head(n).tolist()
        print(f"Loaded {len(tickers)} tickers from {file_path}")
        return tickers

    except Exception as e:
        print(f"Error loading tickers: {e}")
        return []
    
def get_back_up_stock_universe():
    """
    
    Returns a predefined list of tickers as a backup.
    
    """
    major_tickers = [
        'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META', 'BRK-B', 'JNJ', 'JPM',
        'V', 'PG', 'UNH', 'HD', 'DIS', 'PYPL', 'NFLX', 'ADBE', 'CRM', 'INTC',
        'CSCO', 'PEP', 'T', 'ABT', 'TMO', 'COST', 'AVGO', 'LLY', 'WMT', 'XOM',
        'BAC', 'MA', 'CVX', 'ABBV', 'MRK', 'PFE', 'WFC', 'ORCL', 'ACN', 'DHR',
        'NKE', 'PM', 'TXN', 'UNP', 'RTX', 'HON', 'LOW', 'UPS', 'SBUX', 'AMD',
        'IBM', 'SPGI', 'LMT', 'MDT', 'AXP', 'GS', 'BLK', 'SCHW', 'DE', 'CAT',
        'BA', 'MMM', 'GE', 'F', 'GM', 'TWTR', 'SNAP', 'UBER', 'LYFT', 'SHOP',
        'SQ', 'ROKU', 'ZM', 'DOCU', 'CRWD', 'OKTA', 'NET', 'DDOG', 'SNOW', 'PLTR',
        'AI', 'PATH', 'ASAN', 'CLOV', 'LCID', 'RIVN', 'NIO', 'XPEV', 'LI', 
        'FVRR', 'UPST', 'AFRM', 'COIN', 'HOOD', 'SOFI', 'RBLX', 'U', 'DKNG',
        'PTON', 'ABNB', 'EXPE', 'BKNG', 'MAR', 'HLT', 'CMG', 'MCD', 'SBUX',
        'YUM', 'DPZ', 'NCLH', 'CCL', 'RCL', 'AAL', 'DAL', 'LUV', 'UAL',
        'TSM', 'ASML', 'SONY', 'NTES', 'BABA', 'JD', 'PDD', 'BIDU', 'TCEHY',
        'NVO', 'AZN', 'GSK', 'SNY', 'NVS', 'RHHBY', 'PBR', 'VALE', 'BBD',
        'ITUB', 'BSBR', 'SAN', 'BBVA', 'TEVA', 'TLK', 'EBAY', 'PYPL', 'V',
        'MA', 'AXP', 'DFS', 'COF', 'SYF', 'ALLY', 'KEY', 'MTB', 'CFG',
        'WFC', 'JPM', 'BAC', 'C', 'USB', 'TFC', 'PNC', 'BK', 'STT',
        'NDAQ', 'CME', 'ICE', 'MCO', 'SPGI', 'MSCI', 'FICO', 'JKHY',
        'PAYX', 'ADP', 'GWW', 'FAST', 'MSM', 'SNX', 'ARW', 'AVT',
        'TSCO', 'ORLY', 'AZO', 'GPC', 'SNA', 'WSO', 'WCC', 'FAST',
        'MCK', 'ABC', 'CAH', 'HSIC', 'PDCO', 'XRAY', 'ABMD', 'TNDM',
        'DXCM', 'ISRG', 'ALGN', 'IDXX', 'EW', 'BSX', 'BDX', 'SYK',
        'ZBH', 'VAR', 'BAX', 'HCA', 'UHS', 'THC', 'EHC', 'LPNT',
        'SEM', 'STE', 'PODD', 'NVCR', 'IRTC', 'NTRA', 'TWST',
        'VCYT', 'CDNS', 'ANSS', 'DASTY', 'ADSK', 'MTCH', 'TTD',
        'PINS', 'SNAP', 'ROKU', 'ZM', 'PTON', 'DKNG', 'CRSR',
        'LOG', 'FOUR', 'BILL', 'PAYC', 'ESTC', 'MDB', 'DBX',
        'BOX', 'DOCU', 'CLDR', 'DATA', 'TWLO', 'BAND', 'FSLY',
        'NET', 'AKAM', 'ZM', 'TEAM', 'WORK', 'ASAN', 'SMAR',
        'PLAN', 'BASE', 'AI', 'PATH', 'RPD', 'ESTC', 'MDB',
        'DDOG', 'SPLK', 'NEWR', 'APPN', 'APPF', 'CLSK',
        'MSTR', 'COIN', 'RIOT', 'MARA', 'HUT', 'BITF',
        'BTBT', 'CAN', 'GLD', 'SLV', 'USO', 'UNG',
        'TLT', 'IEF', 'LQD', 'HYG', 'JNK', 'EMB',
        'VTI', 'VOO', 'IVV', 'SPY', 'QQQ', 'IWM',
        'DIA', 'VEU', 'VXUS', 'BND', 'AGG', 'LQD',
        'MBB', 'EMB', 'JNK', 'HYG', 'SHY', 'IEI',
        'TLT', 'IEF', 'TIP', 'LTPZ', 'IPE', 'DBC',
        'GSG', 'PDBC', 'COMT', 'FTGC', 'DBA', 'DBB',
        'DBO', 'USO', 'UNG', 'UCO', 'SCO', 'BNO',
        'DBE', 'DBP', 'GLD', 'SLV', 'IAU', 'AGQ',
        'SLV', 'SIVR', 'PSLV', 'PHYS', 'SGOL',
        'GDX', 'GDXJ', 'NUGT', 'JNUG', 'DUST',
        'JDST', 'USLV', 'DSLV', 'UBT', 'TMF',
        'TQQQ', 'SQQQ', 'SPXL', 'SPXS', 'SOXL',
        'SOXS', 'FAS', 'FAZ', 'TNA', 'TZA',
        'LABU', 'LABD', 'YINN', 'YANG', 'RUSL',
        'RUSS', 'ERX', 'ERY', 'GASL', 'GASX',
        'NRGU', 'NRGD', 'WTI', 'USO', 'UCO'
    ]
    print(f"Using backup ticker list with {len(major_tickers)} tickers.")
    return major_tickers

def download_stock_yahoo(ticker, start_date, end_date):
    """
    Download stock data using Yahoo Finance API directly
    
    """
    try:
        # Convert dates to timestamps
        start_ts = int(pd.Timestamp(start_date).timestamp())
        end_ts = int(pd.Timestamp(end_date).timestamp())
        
        # Yahoo Finance API URL
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
        params = {
            'period1': start_ts,
            'period2': end_ts,
            'interval': '1mo',
            'events': 'history'
        }
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, params=params, headers=headers, timeout=10)
        data = response.json()
        
        # Parse the response
        if 'chart' in data and 'result' in data['chart']:
            result = data['chart']['result'][0]
            timestamps = result['timestamp']
            closes = result['indicators']['quote'][0]['close']
            
            # Create DataFrame
            dates = [pd.Timestamp(t, unit='s') for t in timestamps]
            df = pd.DataFrame({'Close': closes}, index=dates)
            df = df.resample('ME').last()  # Ensure monthly frequency
            
            return df['Close']
        else:
            return pd.Series(name=ticker, dtype=float)
            
    except Exception as e:
        print(f"Error downloading {ticker}: {str(e)}")
        return pd.Series(name=ticker, dtype=float)

def main():
    start_date = '2011-07-01'
    end_date = '2024-12-31'
    
    tickers = get_stock_universe('Dissertation/Transfer Learning Data/stock_profile_data.csv')[:645]  
    #tickers = get_back_up_stock_universe()
    print(f"Downloading {len(tickers)} stocks...")
    
    all_data = pd.DataFrame()
    successful = []
    failed = []
    
    for i, ticker in enumerate(tickers):
        print(f"Downloading {ticker} ({i+1}/{len(tickers)})")
        
        series = download_stock_yahoo(ticker, start_date, end_date)
        
        if not series.empty:
            all_data[ticker] = series
            successful.append(ticker)
        else:
            failed.append(ticker)
            # Create empty series with proper date range
            date_range = pd.date_range(start=start_date, end=end_date, freq='ME')
            empty_series = pd.Series(index=date_range, name=ticker, dtype=float)
            all_data[ticker] = empty_series
        
        time.sleep(0.5)  # Rate limiting
    
    # Save to CSV
    all_data.to_csv('monthly_stock_prices_2011_2024.csv')
    
    print(f"Download completed!")
    print(f"Successful: {len(successful)}, Failed: {len(failed)}")
    
    return all_data


data = main()

Use backup to augment the stock universe if the required number of files is not successfully downloaded. 

To do this you can uncomment the line tickers = get_back_up_stock_universe() and comment out tickers = get_stock_universe('Dissertation/Transfer Learning Data/stock_profile_data.csv')[:645]  


In [ ]:
"""Used to remove empty values from the output file"""
import pandas as pd

def remove_empty_columns(input_file, output_file):
    """
    Removes columns that contain only NaN or empty values from a CSV file.
    
    Parameters:
        input_file (str): Path to the original CSV file.
        output_file (str): Path to save the cleaned CSV file.
        
    """
    # Read the CSV file
    df = pd.read_csv(input_file)
    print(f"Original shape: {df.shape}")

    # Drop columns that are completely empty or contain only NaNs
    cleaned_df = df.dropna(axis=1, how='all')

    # Drop columns that contain only empty strings (after stripping whitespace)
    cleaned_df = cleaned_df.loc[:, (cleaned_df.astype(str).apply(lambda x: x.str.strip() != '').any())]

    print(f"Cleaned shape: {cleaned_df.shape}")

    # Save the cleaned CSV
    cleaned_df.to_csv(output_file, index=False)
    print(f"Cleaned file saved to: {output_file}")

if __name__ == "__main__":
    input_csv = "Dissertation/Transfer Learning Data/monthly_stock_prices_fallback_2011_2024.csv"
    output_csv = "cleaned_yfinance_output.csv"
    remove_empty_columns(input_csv, output_csv)

Used to remove empty values from the output file

In [ ]:
"""Time Series Imputation of missing yfinance stock data using time-aware methods and KNN."""

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import warnings
import re
warnings.filterwarnings('ignore')

class StockTimeSeriesImputer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.date_col = 'Date'

    def load_data(self):
        """Load and clean the CSV file"""
        print(f"Loading data from: {self.file_path}")
        self.df = pd.read_csv(self.file_path)

        # Ensure 'Date' column exists
        if self.date_col not in self.df.columns:
            raise ValueError(f"Expected a '{self.date_col}' column in the dataset")

        # Parse dates safely
        self.df[self.date_col] = pd.to_datetime(self.df[self.date_col], errors='coerce')
        if self.df[self.date_col].isna().any():
            print(f"Warning: {self.df[self.date_col].isna().sum()} invalid dates found and set to NaT")

        # Sort chronologically
        self.df = self.df.sort_values(self.date_col).reset_index(drop=True)
        print(f"Data loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
        print(f"Date range: {self.df[self.date_col].min().date()} → {self.df[self.date_col].max().date()}")
        return self.df

    def clean_numeric_values(self):
        """Clean up numeric columns (remove $, commas, etc.)"""
        print("Cleaning numeric columns...")
        for col in self.df.columns:
            if col == self.date_col:
                continue
            self.df[col] = (
                self.df[col]
                .astype(str)
                .replace(r'[^\d.\-]', '', regex=True)
                .replace('', np.nan)
                .astype(float)
            )
        print("Numeric cleaning complete.")
        return self.df

    def create_time_features(self):
        """Add simple time index to assist interpolation"""
        self.df = self.df.sort_values(self.date_col).reset_index(drop=True)
        self.df['time_index'] = np.arange(len(self.df))
        return self.df

    def simple_imputation(self):
        """Perform time-aware imputation with multiple methods"""
        print("Starting simple imputation...")
        df_imputed = self.df.copy()
        numeric_cols = [col for col in df_imputed.columns if col not in [self.date_col, 'time_index']]

        # Step 1: Forward fill
        df_imputed[numeric_cols] = df_imputed[numeric_cols].fillna(method='ffill')

        # Step 2: Backward fill
        df_imputed[numeric_cols] = df_imputed[numeric_cols].fillna(method='bfill')

        # Step 3: Linear interpolation (handles middle gaps)
        df_imputed[numeric_cols] = df_imputed[numeric_cols].interpolate(method='linear', limit_direction='both')

        # Step 4: Median fallback for any remaining missing
        for col in numeric_cols:
            if df_imputed[col].isna().any():
                df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

        print("Simple imputation complete.")
        return df_imputed

    def advanced_imputation(self, df_partial):
        """Apply KNN imputation for more robust completion"""
        print("Running KNN imputation...")
        numeric_cols = [col for col in df_partial.columns if col not in [self.date_col, 'time_index']]
        knn = KNNImputer(n_neighbors=3, weights='distance')
        df_partial[numeric_cols] = knn.fit_transform(df_partial[numeric_cols])
        print("KNN imputation complete.")
        return df_partial

    def validate(self, original_df, imputed_df):
        """Validate imputation effectiveness"""
        missing_before = original_df.isna().sum().sum()
        missing_after = imputed_df.isna().sum().sum()
        print("\nValidation Report")
        print("=" * 30)
        print(f"Missing before: {missing_before}")
        print(f"Missing after:  {missing_after}")
        if missing_before > 0:
            print(f"Filled {missing_before - missing_after} values ({(1 - missing_after/missing_before) * 100:.2f}% success)")
        else:
            print("No missing values to impute.")

    def save(self, df_imputed, output_path):
        """Save the imputed dataset"""
        final_df = df_imputed.drop(columns=['time_index'], errors='ignore')
        final_df.to_csv(output_path, index=False)
        print(f"Imputed data saved to: {output_path}")
        return final_df

    def run_pipeline(self, output_path='imputed_yfinance_data.csv'):
        """Full pipeline"""
        print("Starting stock data imputation pipeline...")
        self.load_data()
        self.clean_numeric_values()
        self.create_time_features()
        df_simple = self.simple_imputation()
        df_final = self.advanced_imputation(df_simple)
        self.validate(self.df, df_final)
        return self.save(df_final, output_path)


if __name__ == "__main__":
    file_path = "Dissertation/Transfer Learning Data/cleaned_yfinance_output.csv"  # replace with your actual path
    imputer = StockTimeSeriesImputer(file_path)
    imputed_data = imputer.run_pipeline("imputed_yfinance_data.csv")


In [ ]:
"""Imputation of missing ONS data using time-aware methods and KNN."""

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import warnings
import re
warnings.filterwarnings('ignore')

class TimeSeriesImputer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.date_col = 'Date'

    def load_data(self):
        """Load and clean the CSV file"""
        print(f"Loading data from: {self.file_path}")
        self.df = pd.read_csv(self.file_path)

        # Ensure 'Date' column exists
        if self.date_col not in self.df.columns:
            raise ValueError(f"Expected a '{self.date_col}' column in the dataset")

        # Parse dates safely
        self.df[self.date_col] = pd.to_datetime(self.df[self.date_col], errors='coerce')
        if self.df[self.date_col].isna().any():
            print(f"Warning: {self.df[self.date_col].isna().sum()} invalid dates found and set to NaT")

        # Sort chronologically
        self.df = self.df.sort_values(self.date_col).reset_index(drop=True)
        print(f"Data loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
        print(f"Date range: {self.df[self.date_col].min().date()} → {self.df[self.date_col].max().date()}")
        return self.df

    def clean_numeric_values(self):
        """Clean up numeric columns (remove $, commas, etc.)"""
        print("Cleaning numeric columns...")
        for col in self.df.columns:
            if col == self.date_col:
                continue
            self.df[col] = (
                self.df[col]
                .astype(str)
                .replace(r'[^\d.\-]', '', regex=True)
                .replace('', np.nan)
                .astype(float)
            )
        print("Numeric cleaning complete.")
        return self.df

    def create_time_features(self):
        """Add simple time index to assist interpolation"""
        self.df = self.df.sort_values(self.date_col).reset_index(drop=True)
        self.df['time_index'] = np.arange(len(self.df))
        return self.df

    def simple_imputation(self):
        """Perform time-aware imputation with multiple methods"""
        print("Starting simple imputation...")
        df_imputed = self.df.copy()
        numeric_cols = [col for col in df_imputed.columns if col not in [self.date_col, 'time_index']]

        # Step 1: Forward fill
        df_imputed[numeric_cols] = df_imputed[numeric_cols].fillna(method='ffill')

        # Step 2: Backward fill
        df_imputed[numeric_cols] = df_imputed[numeric_cols].fillna(method='bfill')

        # Step 3: Linear interpolation (handles middle gaps)
        df_imputed[numeric_cols] = df_imputed[numeric_cols].interpolate(method='linear', limit_direction='both')

        # Step 4: Median fallback for any remaining missing
        for col in numeric_cols:
            if df_imputed[col].isna().any():
                df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

        print("Simple imputation complete.")
        return df_imputed

    def advanced_imputation(self, df_partial):
        """Apply KNN imputation for more robust completion"""
        print("Running KNN imputation...")
        numeric_cols = [col for col in df_partial.columns if col not in [self.date_col, 'time_index']]
        knn = KNNImputer(n_neighbors=3, weights='distance')
        df_partial[numeric_cols] = knn.fit_transform(df_partial[numeric_cols])
        print("KNN imputation complete.")
        return df_partial

    def validate(self, original_df, imputed_df):
        """Validate imputation effectiveness"""
        missing_before = original_df.isna().sum().sum()
        missing_after = imputed_df.isna().sum().sum()
        print("\nValidation Report")
        print("=" * 30)
        print(f"Missing before: {missing_before}")
        print(f"Missing after:  {missing_after}")
        if missing_before > 0:
            print(f"Filled {missing_before - missing_after} values ({(1 - missing_after/missing_before) * 100:.2f}% success)")
        else:
            print("No missing values to impute.")

    def save(self, df_imputed, output_path):
        """Save the imputed dataset"""
        final_df = df_imputed.drop(columns=['time_index'], errors='ignore')
        final_df.to_csv(output_path, index=False)
        print(f"Imputed data saved to: {output_path}")
        return final_df

    def run_pipeline(self, output_path='imputed_ONS_data.csv'):
        """Full pipeline"""
        print("Starting stock data imputation pipeline...")
        self.load_data()
        self.clean_numeric_values()
        self.create_time_features()
        df_simple = self.simple_imputation()
        df_final = self.advanced_imputation(df_simple)
        self.validate(self.df, df_final)
        return self.save(df_final, output_path)


if __name__ == "__main__":
    file_path = "Dissertation/Transfer Learning Data/ONS Monthly gross domestic product.csv"  # replace with your actual path
    imputer = TimeSeriesImputer(file_path)
    imputed_data = imputer.run_pipeline("imputed_ONS_data.csv")


In [ ]:
"""Processor for financial CSV files FROM FRED."""

import pandas as pd
import os
import glob
from pathlib import Path

def financial_data_processor(folder_path, output_1975='financial_data_1975.csv', 
                                    output_2000='financial_data_2000.csv'):
    """
    Processor for financial CSV files with better error handling and data validation.
    
    """
    
    # Get all CSV files
    csv_files = list(Path(folder_path).glob("*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in {folder_path}")
        return None, None
    
    print(f"Found {len(csv_files)} CSV files")
    
    all_data = []
    file_info = []
    
    # First pass: read all files and collect information
    for file_path in csv_files:
        try:
            # Try different encodings
            for encoding in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    df = pd.read_csv(file_path, encoding=encoding)
                    break
                except UnicodeDecodeError:
                    continue
            else:
                print(f" Could not read {file_path.name} with common encodings")
                continue
            
            if df.empty:
                continue
            
            # Find date column
            date_col = None
            for col in df.columns:
                if any(keyword in col.lower() for keyword in ['date', 'time', 'day']):
                    date_col = col
                    break
            
            if not date_col:
                print(f"  No date column found in {file_path.name}")
                continue
            
            # Convert date column
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
            df = df.dropna(subset=[date_col])
            
            if df.empty:
                continue
            
            df.set_index(date_col, inplace=True)
            
            # Store file info for processing
            file_info.append({
                'df': df,
                'name': file_path.stem,
                'numeric_cols': df.select_dtypes(include=['number']).columns.tolist()
            })
            
            print(f"  {file_path.name}: {len(df)} rows, {len(df.columns)} columns")
            
        except Exception as e:
            print(f" Error with {file_path.name}: {str(e)}")
            continue
    
    if not file_info:
        print("No valid data found in any CSV files")
        return None, None
    
    # Create complete date ranges
    all_dates = set()
    for info in file_info:
        all_dates.update(info['df'].index)
    
    if not all_dates:
        print("No valid dates found")
        return None, None
    
    min_date = min(all_dates)
    max_date = max(all_dates)
    
    # Create master DataFrames
    full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
    master_df = pd.DataFrame(index=full_date_range)
    
    # Merge all data
    for info in file_info:
        df = info['df']
        file_name = info['name']
        
        for col in info['numeric_cols']:
            col_name = f"{file_name}_{col}" if len(info['numeric_cols']) > 1 else file_name
            master_df[col_name] = df[col]
    
    # Create filtered outputs
    start_1975 = pd.Timestamp('1975-01-01')
    start_2000 = pd.Timestamp('2000-01-01')
    
    # 1975 output
    df_1975 = master_df[master_df.index >= start_1975].copy()
    if not df_1975.empty:
        df_1975.to_csv(output_1975)
        print(f"\n1975 output: {len(df_1975)} rows, {len(df_1975.columns)} columns")
        print(f"Date range: {df_1975.index.min().date()} to {df_1975.index.max().date()}")
    
    # 2000 output
    df_2000 = master_df[master_df.index >= start_2000].copy()
    if not df_2000.empty:
        df_2000.to_csv(output_2000)
        print(f"2000 output: {len(df_2000)} rows, {len(df_2000.columns)} columns")
        print(f"Date range: {df_2000.index.min().date()} to {df_2000.index.max().date()}")
    
    return df_1975, df_2000


if __name__ == "__main__":
    folder_path = input("Enter folder path containing CSV files: ").strip('"\'')
    
    if os.path.exists(folder_path):
        df_1975, df_2000 = financial_data_processor(folder_path)
        
        if df_1975 is not None:
            print(f"\nOutput files created successfully!")
            print(f"1975 data: financial_data_1975.csv")
            print(f"2000 data: financial_data_2000.csv")
        else:
            print("Failed to process data")
    else:
        print(f"Folder '{folder_path}' does not exist")

In [ ]:
"""Advanced Hybrid Imputation script to mitigate missing time series data using multiple techniques including LSTM."""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer
from statsmodels.tsa.statespace.kalman_smoother import KalmanSmoother

# Try import TF for LSTM;  degrade if unavailable
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models, callbacks, backend as K
    TF_AVAILABLE = True
except Exception:
    TF_AVAILABLE = False


class HybridTimeSeriesImputerWithLSTM:
    def __init__(self, file_path, drop_threshold=0.9):
        self.file_path = file_path
        self.df = None
        self.date_col = 'Date'
        self.drop_threshold = drop_threshold

    # Load / Clean
    def load_data(self):
        print(f" Loading: {self.file_path}")
        self.df = pd.read_csv(self.file_path)
        if self.date_col not in self.df.columns:
            raise ValueError(f"Expected '{self.date_col}' column")

        self.df[self.date_col] = pd.to_datetime(self.df[self.date_col], errors='coerce')
        self.df = self.df.sort_values(self.date_col).reset_index(drop=True)
        print(f" Loaded {self.df.shape[0]} rows × {self.df.shape[1]} cols")
        return self.df

    def clean_numeric_values(self):
        print("Cleaning numeric columns (robust float handling)...")
        import re
        def safe_float(x):
            if pd.isna(x): return np.nan
            s = str(x).strip()
            if s in ['', 'nan', 'NaN', 'None', 'null']: return np.nan
            # fix malformed sci notation like -9.08-05 -> -9.08e-05
            s = re.sub(r'(-?\d*\.?\d*)-+(\d+)$', r'\1e-\2', s)
            s = re.sub(r'[^\deE\.\-\+]', '', s)
            try:
                return float(s)
            except:
                return np.nan

        for col in self.df.columns:
            if col == self.date_col: continue
            self.df[col] = self.df[col].map(safe_float)

        print("Numeric cleaning done.")
        return self.df

    def drop_sparse_columns(self):
        missing_fraction = self.df.isna().mean()
        sparse = missing_fraction[missing_fraction > self.drop_threshold].index.tolist()
        if sparse:
            print(f"Dropping {len(sparse)} sparse columns (> {int(self.drop_threshold*100)}% missing)")
            self.df = self.df.drop(columns=sparse)
        return self.df

    def create_time_features(self):
        self.df['time_index'] = np.arange(len(self.df))
        return self.df

    # Simple time-aware 
    def simple_imputation(self):
        print("Simple time-aware imputation (ffill, bfill, linear, time)...")
        df_imp = self.df.copy()
        numeric_cols = [c for c in df_imp.columns if c not in [self.date_col, 'time_index']]

        # forward/backward, linear
        df_imp[numeric_cols] = df_imp[numeric_cols].fillna(method='ffill').fillna(method='bfill')
        df_imp[numeric_cols] = df_imp[numeric_cols].interpolate(method='linear', limit_direction='both')

        # time-based interpolation if date index is usable
        try:
            df_imp.set_index(self.date_col, inplace=True)
            df_imp[numeric_cols] = df_imp[numeric_cols].interpolate(method='time')
            df_imp.reset_index(inplace=True)
        except Exception:
            pass

        # median fallback
        for c in numeric_cols:
            if df_imp[c].isna().any():
                df_imp[c] = df_imp[c].fillna(df_imp[c].median())

        print("Simple imputation done.")
        return df_imp

    #  KNN imputation
    def knn_imputation(self, df_partial, n_neighbors=5):
        print(f"KNN imputation (k={n_neighbors})...")
        numeric_cols = [c for c in df_partial.columns if c not in [self.date_col, 'time_index']]
        imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
        df_partial[numeric_cols] = imputer.fit_transform(df_partial[numeric_cols])
        print("KNN done.")
        return df_partial

    #Iterative (MICE)
    def iterative_imputation(self, df_partial, max_iter=10):
        print("IterativeImputer (MICE)...")
        numeric_cols = [c for c in df_partial.columns if c not in [self.date_col, 'time_index']]
        imp = IterativeImputer(max_iter=max_iter, random_state=0)
        df_partial[numeric_cols] = imp.fit_transform(df_partial[numeric_cols])
        print("Iterative imputation done.")
        return df_partial

    # Kalman smoother 
    def kalman_smoothing(self, df_partial):
        print("Kalman smoothing (per-column true fill engine)...")
        df_sm = df_partial.copy()
        numeric_cols = [c for c in df_sm.columns if c not in [self.date_col, 'time_index']]
        total_filled = 0

        for col in numeric_cols:
            series = df_sm[col].astype(float).values.squeeze()
            if np.isnan(series).all():
                continue
            nan_mask = np.isnan(series)
            if nan_mask.sum() == 0:
                continue

            filled_series = pd.Series(series).interpolate(limit_direction='both').values

            try:
                kf = KalmanSmoother(k_endog=1, k_states=1)
                kf.bind(filled_series)
                # initialize with correct shapes
                kf.initialize_known(np.array([filled_series[0]]), np.array([[1.0]]))
                kf['transition'] = np.array([[1.0]])
                kf['design'] = np.array([[1.0]])
                kf['selection'] = np.array([[1.0]])
                kf['state_cov'] = np.array([[0.001]])
                kf['obs_cov'] = np.array([[0.01]])

                result = kf.smooth(filled_series)
                smoothed = result.smoothed_state[0].flatten()

                # Fill missing points using smoothed state
                series[nan_mask] = smoothed[nan_mask]
                df_sm[col] = series

                filled = nan_mask.sum() - np.isnan(series).sum()
                total_filled += filled
                print(f"  {col}: filled {filled} / {nan_mask.sum()} missing")

            except Exception as e:
                # fallback to linear interpolation if Kalman fails (keeps moving pipeline)
                df_sm[col] = pd.Series(filled_series).interpolate(method='linear', limit_direction='both').values
                print(f"  {col}: Kalman failed ({e}) -> linear fallback")

        print(f"Kalman smoothing summary: total_filled={total_filled:,}")
        return df_sm

    #  LSTM-based refinement 
    def lstm_refinement(self, df_partial, seq_len=12, epochs=25, batch_size=32, min_train_samples=100):
        """
        Try to learn temporal + cross-series relations and iteratively fill remaining missing values.
        - For each column with missing values: build sequences from all numeric columns (multivariate)
        - Train a small LSTM to predict the next value of target column
        - Iteratively forecast missing runs using recent context (allowing the model to feed its own predictions)
        """
        if not TF_AVAILABLE:
            print("TensorFlow not available — skipping LSTM refinement.")
            return df_partial

        print(" Starting LSTM refinement (may be slower) ...")
        df_lstm = df_partial.copy()
        numeric_cols = [c for c in df_lstm.columns if c not in [self.date_col, 'time_index']]

        # Prepare normalized scaler per column for stability (simple z-score)
        col_means = df_lstm[numeric_cols].mean()
        col_stds = df_lstm[numeric_cols].std().replace(0, 1.0)

        # normalized copy for model training
        norm = (df_lstm[numeric_cols] - col_means) / col_stds

        # For each column with missing values, attempt to train and fill
        for target_col in numeric_cols:
            mask = df_lstm[target_col].isna()
            missing_count = mask.sum()
            if missing_count == 0:
                continue

            # Build supervised sequences where next-step target exists and no NaNs in window
            X, y = [], []
            arr = norm.values  # shape (T, F)
            T, F = arr.shape

            for i in range(seq_len, T):
                window = arr[i-seq_len:i, :]          # seq_len x F
                target_val = arr[i, numeric_cols.index(target_col)]
                # require that window and target are finite (no NaNs)
                if np.isfinite(window).all() and np.isfinite(target_val):
                    X.append(window)
                    y.append(target_val)

            X = np.array(X)
            y = np.array(y)

            if len(y) < min_train_samples:
                print(f" {target_col}: insufficient training samples ({len(y)}). Skipping LSTM for this column.")
                continue

            # Build simple LSTM model
            K.clear_session()
            model = models.Sequential([
                layers.Input(shape=(seq_len, F)),
                layers.LSTM(64, return_sequences=False),
                layers.Dense(32, activation='relu'),
                layers.Dense(1)
            ])
            model.compile(optimizer='adam', loss='mse')

            # Early stopping
            es = callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

            # Train
            model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0, callbacks=[es])

            # Iteratively fill missing indices in chronological order if enough context exists
            idxs = np.where(mask)[0]
            filled_this_col = 0
            arr_copy = arr.copy()

            for idx in idxs:
                # Need seq_len most recent steps before idx to be available (use arr_copy which contains filled preds)
                if idx - seq_len < 0:
                    continue

                window = arr_copy[idx-seq_len:idx, :]
                # require that window has no NaN
                if not np.isfinite(window).all():
                    continue

                inp = window.reshape(1, seq_len, F)
                pred = model.predict(inp, verbose=0)[0, 0]  # normalized prediction
                # write back into arr_copy and df_lstm (denormalize)
                denorm = pred * col_stds[target_col] + col_means[target_col]
                arr_copy[idx, numeric_cols.index(target_col)] = pred
                df_lstm.at[idx, target_col] = denorm
                filled_this_col += 1

            print(f" {target_col}: LSTM filled {filled_this_col} / {missing_count} missing")
        print("LSTM refinement complete.")
        return df_lstm

    # Final median fallback (guaranteee no NaNs)
    def final_median_fallback(self, df_partial):
        print("Final median fallback — ensuring no NaNs remain ...")
        df_final = df_partial.copy()
        numeric_cols = [c for c in df_final.columns if c not in [self.date_col, 'time_index']]
        for col in numeric_cols:
            if df_final[col].isna().any():
                med = df_final[col].median()
                df_final[col] = df_final[col].fillna(med)
                print(f"  → {col}: filled remaining NaNs with median {med:.6g}")
        remaining = df_final.isna().sum().sum()
        print(f" Remaining missing after median fallback: {remaining:,}")
        return df_final

    #  Validate 
    def validate(self, original_df, imputed_df):
        missing_before = original_df.isna().sum().sum()
        missing_after = imputed_df.isna().sum().sum()
        print("\nValidation Report")
        print("="*40)
        print(f"Missing before: {missing_before:,}")
        print(f"Missing after:  {missing_after:,}")
        if missing_before > 0:
            print(f"Filled {missing_before - missing_after:,} values ({(1 - missing_after / missing_before) * 100:.2f}% success)")
        else:
            print("No missing values originally.")

    #  Save 
    def save(self, df_final, output_path):
        df_final.drop(columns=['time_index'], errors='ignore').to_csv(output_path, index=False)
        print(f"Saved: {output_path}")
        return df_final

    # Full pipeline 
    def run_pipeline(self, output_path='imputed_hybrid_lstm.csv', lstm_params=None):
        print("\n Starting full pipeline (with LSTM refinement)...")
        self.load_data()
        self.clean_numeric_values()
        self.drop_sparse_columns()
        self.create_time_features()

        df_simple = self.simple_imputation()
        df_knn = self.knn_imputation(df_simple)
        df_iter = self.iterative_imputation(df_knn)
        df_kal = self.kalman_smoothing(df_iter)

        # LSTM params
        if lstm_params is None:
            lstm_params = dict(seq_len=12, epochs=25, batch_size=32, min_train_samples=100)

        df_lstm = self.lstm_refinement(df_kal, **lstm_params)
        df_final = self.final_median_fallback(df_lstm)

        self.validate(self.df, df_final)
        return self.save(df_final, output_path)


if __name__ == "__main__":
    file_path = "C:/Users/Bella/Documents/Dissertation/Transfer Learning Data/financial_data_2000.csv"
    imputer = HybridTimeSeriesImputerWithLSTM(file_path)
    imputed_data = imputer.run_pipeline("imputed_2000_FRED_data_hybrid.csv", lstm_params=dict(seq_len=12, epochs=30, batch_size=32, min_train_samples=200))



Imputation for both the data files created by the former script

In [ ]:
"""Convert daily financial imputated FRED data to monthly averages."""
import pandas as pd

def daily_to_monthly_average(input_file, output_file):
    # Read the CSV
    df = pd.read_csv(input_file)

    # Ensure date column is datetime
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

    # Drop any rows with invalid dates
    df = df.dropna(subset=['Date'])

    # Set Date as index for resampling
    df.set_index('Date', inplace=True)

    # Compute monthly averages
    monthly_avg = df.resample('M').mean()

    # Reset index and format month as YYYY-MM
    monthly_avg.reset_index(inplace=True)
    monthly_avg['Date'] = monthly_avg['Date'].dt.strftime('%Y-%m')

    # Save to CSV
    monthly_avg.to_csv(output_file, index=False)
    print(f" Monthly averages saved to {output_file}")

if __name__ == "__main__":
    input_path = "imputed_1975_FRED_data_hybrid.csv"   # change this to your input file path
    output_path = "monthly_averages_FRED_1975.csv"
    daily_to_monthly_average(input_path, output_path)


In [ ]:
"""John Hopkins Covid-19 Data Transposition Script"""
import pandas as pd

def transpose_csv(input_file, output_file):
    # Read CSV file
    df = pd.read_csv(input_file)

    # Assume first column is the row header (e.g. 'Country/Region')
    row_header = df.columns[0]

    # Set 'Country/Region' column as index
    df.set_index(row_header, inplace=True)

    # Transpose the DataFrame (swap rows and columns)
    df_transposed = df.transpose()

    # Reset the index so dates become a proper column again
    df_transposed.reset_index(inplace=True)

    # Rename the index column to 'Date' (can adjust if needed)
    df_transposed.rename(columns={'index': 'Date'}, inplace=True)

    # Save to new CSV
    df_transposed.to_csv(output_file, index=False)
    print(f"Transposed CSV saved to {output_file}")


if __name__ == "__main__":
    input_path = "Dissertation/Transfer Learning Data/john_hopkins_time_series_covid19_deaths_global.csv"   # change this to your input file path
    output_path = "transposed_JH_covid_deaths_global.csv"
    transpose_csv(input_path, output_path)


In [ ]:
"""University of Maryland Global Terrorism Database Monthly Attack Summary with Missing Months Filled"""
import pandas as pd
import calendar

def monthly_attack_summary_fill_missing(input_file, output_file):
    # Read input CSV
    df = pd.read_csv(input_file)

    # Ensure required columns exist
    required_cols = ['iyear', 'imonth', 'country_txt', 'attacktype1_txt']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Filter invalid or missing months (should be 0–11)
    df = df[df['imonth'].between(0, 11)]

    # Map numeric months (0–11) to month names
    month_names = {i: calendar.month_name[i + 1] for i in range(12)}
    df['Date'] = df.apply(lambda x: f"{month_names[x['imonth']]} {int(x['iyear'])}", axis=1)

    # Create attack_type_country column
    df['attack_type_country'] = df['attacktype1_txt'] + '_' + df['country_txt']

    # Group by Date and attack_type_country
    grouped = (
        df.groupby(['Date', 'attack_type_country'])
          .size()
          .reset_index(name='count')
    )

    # Determine full year range in dataset
    years = sorted(df['iyear'].unique())

    # Build full list of "Month Year" combinations
    all_dates = [f"{calendar.month_name[m+1]} {y}" for y in years for m in range(12)]

    # Pivot to wide format
    pivot_df = grouped.pivot_table(
        index='Date',
        columns='attack_type_country',
        values='count',
        fill_value=0
    ).reset_index()

    # Ensure all months/years are present (fill missing with zeros)
    pivot_df = pivot_df.set_index('Date').reindex(all_dates, fill_value=0).reset_index()

    # Sort columns alphabetically, keeping "Date" first
    cols = ['Date'] + sorted([c for c in pivot_df.columns if c != 'Date'])
    pivot_df = pivot_df[cols]

    # Save output
    pivot_df.to_csv(output_file, index=False)
    print(f"Monthly attack summary (with missing months filled) saved to {output_file}")


if __name__ == "__main__":
    input_path = "Dissertation/Transfer Learning Data/globalterrorismdb_0522dist.csv"   # change to your input file path
    output_path = "monthly_attack_summary_global_terrorism_database.csv"
    monthly_attack_summary_fill_missing(input_path, output_path)


In [ ]:
import os
import csv
import numpy as np

# Smoothing Functions

def exponential_smoothing(series, alpha):
    result = [series[0]]
    for n in range(1, len(series)):
        result.append(alpha * series[n] + (1 - alpha) * result[n-1])
    return result

def double_exponential_smoothing(series, alpha, beta):
    result = [series[0]]
    for n in range(1, len(series)+1):
        if n == 1:
            level, trend = series[0], series[1] - series[0]
        if n >= len(series):  
            value = result[-1]
        else:
            value = series[n]
        last_level, level = level, alpha * value + (1 - alpha) * (level + trend)
        trend = beta * (level - last_level) + (1 - beta) * trend
        result.append(level + trend)
    return result[:-1]  # remove the forecast extra row


#   Process All CSVs in Folder

input_folder = "Dissertation/Transfer Learning/Transfer Learning Data/for use"           # folder containing CSVs
alpha = 0.1
beta = 0.3

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(".csv"):
        continue

    filepath = os.path.join(input_folder, filename)
    print(f"Processing: {filepath}")

    # Load CSV
    with open(filepath, "r", newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)

    header = rows[0]            # keep column names
    data = rows[1:]             # keep month row labels
    month_labels = [row[0] for row in data]

    # Extract numeric values (columns 1..n)
    numeric_data = np.array([row[1:] for row in data], dtype=float)

    # Apply smoothing column-by-column
    smoothed_cols = []
    for col in numeric_data.T:
        smoothed_col = double_exponential_smoothing(col, alpha, beta)
        smoothed_cols.append(smoothed_col)

    smoothed_matrix = np.array(smoothed_cols).T

    # Reassemble final table
    output_rows = [header]
    for i, month in enumerate(month_labels):
        output_rows.append([month] + list(smoothed_matrix[i]))

    # Save output file
    outname = f"sm_g{filename}"
    outpath = os.path.join(input_folder, outname)

    with open(outpath, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(output_rows)

    print(f"Saved to {outpath}")

print("Done.")
